## RAG Pipeline with Persistent ChromaDB and Query Expansion

### CRUD Operations Creating Vector DB

In [2]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime
# from google.colab import userdata

c:\Users\Vikash Kumar\Desktop\AG0847\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Create


In [3]:
pdf_folder_location = "tesla-annual-reports"

In [4]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [5]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [6]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1124,
    chunk_overlap=101
)

In [7]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [8]:
len(tesla_10k_chunks)

1906

In [117]:
print(tesla_10k_chunks[0])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA

In [10]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

# Database Creation

In [9]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [11]:

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Vikash Kumar\AppData\Local\Temp\ipykernel_2748\2427895705.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [12]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
    
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [36]:
chromadb_client.heartbeat()

1780372451887601500

In [15]:
chromadb_client.count_collections()

2

# Create a collection

In [13]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [130]:
chromadb_client.count_collections()

2

In [131]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023', 'full_document_chunks']

In [21]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(2) # Pause for 30 seconds to avoid rate limiting issues with the vector store

# Read

In [41]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


# Inspecting Individual Records

In [42]:
collection = chromadb_client.get_collection(tesla_10k_collection)

In [43]:
# Count the number of records in the collection
collection.count()

3337

In [44]:
# Inspect the first 10 records
collection.peek()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['text_0',
  'text_1',
  'text_2',
  'text_3',
  'text_4',
  'text_5',
  'text_6',
  'text_7',
  'text_8',
  'text_9'],
 'embeddings': array([[ 0.02675908,  0.00757637, -0.04422267, ..., -0.03966741,
         -0.09228931, -0.0373573 ],
        [ 0.01642387, -0.06928593, -0.02833171, ..., -0.01646161,
         -0.06955732, -0.01540411],
        [ 0.00760543, -0.01997787, -0.00596519, ..., -0.0130257 ,
         -0.06449894, -0.02837417],
        ...,
        [ 0.03221607,  0.0295096 , -0.04174366, ..., -0.02659888,
         -0.04579795,  0.0019708 ],
        [ 0.05646931,  0.12058288, -0.0257278 , ..., -0.05004434,
         -0.03446646, -0.00241624],
        [ 0.0119407 ,  0.02874993, -0.01575188, ...,  0.01867358,
         -0.0381767 ,  0.00357348]]),
 'documents': ['UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K/A\n(Amendment\tNo.\t1)\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES

In [45]:
# tables / keys present in the vector DB
collection.peek().keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'data', 'metadatas', 'included'])

In [46]:
# Inspect a specific record

collection.get(
    ids=['text_999']
)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['text_999'],
 'embeddings': None,
 'documents': ['term,\tassuming\tall\tother\trevenue\trecognition\tcriteria\thave\tbeen\tmet.\tThe\tdifference\tbetween\tthe\tpayments\treceived\tand\tthe\trevenue\trecognized\tis\trecorded\tas\ndeferred\trevenue\tor\tdeferred\tasset\ton\tthe\tconsolidated\tbalance\tsheet.\nFor\tsolar\tenergy\tsystems\twhere\tcustomers\tpurchase\telectricity\tfrom\tus\tunder\tPPAs\tprior\tto\tJanuary\t1,\t2019,\twe\thave\tdetermined\tthat\tthese\tagreements\nshould\tbe\taccounted\tfor\tas\toperating\tleases\tpursuant\tto\tASC\t840.\tRevenue\tis\trecognized\tbased\ton\tthe\tamount\tof\telectricity\tdelivered\tat\trates\tspecified\tunder\nthe\tcontracts,\tassuming\tall\tother\trevenue\trecognition\tcriteria\tare\tmet.\nWe\trecord\tas\tdeferred\trevenue\tany\tamounts\tthat\tare\tcollected\tfrom\tcustomers,\tincluding\tlease\tprepayments,\tin\texcess\tof\trevenue\trecognized\tand\noperations\tand\tmaintenance\tservice\tfees,\twhich\tis\trecognized\tas\trevenue\tra

## Retrieving relevant records

In [47]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [48]:
user_query = "Automotive revenue in 2021?"

In [49]:
retriever.invoke(user_query)

[Document(id='text_3106', metadata={'creationdate': '2024-01-29T11:11:14+00:00', 'creator': 'wkhtmltopdf 0.12.6', 'page': 38, 'page_label': '39', 'producer': 'Qt 5.15.2', 'source': 'tesla-annual-reports\\tsla-20231231-gen.pdf', 'title': '', 'total_pages': 130}, page_content='2022\n2021\n$\n%\n$\n%\nAutomotive\tsales\n$\n78,509\t\n$\n67,210\t\n$\n44,125\t\n$\n11,299\t\n17\t\n%\n$\n23,085\t\n52\t\n%\nAutomotive\tregulatory\tcredits\n1,790\t\n1,776\t\n1,465\t\n14\t\n1\t\n%\n311\t\n21\t\n%\nAutomotive\tleasing\n2,120\t\n2,476\t\n1,642\t\n(356)\n(14)\n%\n834\t\n51\t\n%\nTotal\tautomotive\trevenues\n82,419\t\n71,462\t\n47,232\t\n10,957\t\n15\t\n%\n24,230\t\n51\t\n%\nServices\tand\tother\n8,319\t\n6,091\t\n3,802\t\n2,228\t\n37\t\n%\n2,289\t\n60\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\nrevenue\n90,738\t\n77,553\t\n51,034\t\n13,185\t\n17\t\n%\n26,519\t\n52\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\trevenue\n6,035\t\n3,909\t\n2,789\t\n2,126\t\n54\t\n%\n1,120\t\n40\t\n%\nT

In [50]:
len(retriever.invoke(user_query))

5

# Update

In [51]:
record_index_for_update = 999

In [ ]:
vectorstore_persisted.update_document(
    document_id='text_' + str(record_index_for_update),
    document=Document(
        page_content=tesla_10k_chunks[record_index_for_update].page_content,
        metadata={"creator": "johndoe"}
    )
)

In [53]:
# Inspect a specific record

collection.get(
    ids=['text_999']
)

{'ids': ['text_999'],
 'embeddings': None,
 'documents': ['term,\tassuming\tall\tother\trevenue\trecognition\tcriteria\thave\tbeen\tmet.\tThe\tdifference\tbetween\tthe\tpayments\treceived\tand\tthe\trevenue\trecognized\tis\trecorded\tas\ndeferred\trevenue\tor\tdeferred\tasset\ton\tthe\tconsolidated\tbalance\tsheet.\nFor\tsolar\tenergy\tsystems\twhere\tcustomers\tpurchase\telectricity\tfrom\tus\tunder\tPPAs\tprior\tto\tJanuary\t1,\t2019,\twe\thave\tdetermined\tthat\tthese\tagreements\nshould\tbe\taccounted\tfor\tas\toperating\tleases\tpursuant\tto\tASC\t840.\tRevenue\tis\trecognized\tbased\ton\tthe\tamount\tof\telectricity\tdelivered\tat\trates\tspecified\tunder\nthe\tcontracts,\tassuming\tall\tother\trevenue\trecognition\tcriteria\tare\tmet.\nWe\trecord\tas\tdeferred\trevenue\tany\tamounts\tthat\tare\tcollected\tfrom\tcustomers,\tincluding\tlease\tprepayments,\tin\texcess\tof\trevenue\trecognized\tand\noperations\tand\tmaintenance\tservice\tfees,\twhich\tis\trecognized\tas\trevenue\tra

# Delete

In [ ]:
# The following line deletes records and should be used with caution
vectorstore.delete(ids=['text_996'])

In [55]:

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [56]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [57]:
chromadb_client

In [58]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [59]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [60]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [61]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000028C4B983DA0>, search_kwargs={'k': 5})

# RAG Q&A

In [62]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [63]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [ ]:
user_query = "What was the automotive revenue in 2022?"

In [65]:
relevant_document_chunks = retriever.invoke(user_query)

In [66]:
len(relevant_document_chunks)

5

# Composing the response

In [14]:
from groq import Groq
client = Groq()

In [67]:
user_query = "What was the automotive revenue in 2021?"

In [70]:

model_name = 'openai/gpt-oss-120b'

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

The automotive revenue in 2021 was $47,232 million.


# Improving RAG

In [71]:
import os
import chromadb

from langchain_chroma import Chroma
# from langchain_groq import ChatGroq



from langchain_core.documents import Document

# from google.colab import userdata

In [72]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [73]:
# !pip install langchain-community==0.3.19

from langchain_community.embeddings import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [22]:
from groq import Groq
client=Groq()

In [23]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [76]:
chromadb_client

In [77]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [78]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [79]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [80]:
documents = [
    Document(
        id=1,
        page_content="We design, develop, manufacture, sell and lease high-performance fully electric vehicles and energy generation and storage systems, and offer services related to our products. We generally sell our products directly to customers, and continue to grow our customer-facing infrastructure through a global network of vehicle showrooms and service centers, Mobile Service, body shops, Supercharger stations and Destination Chargers to accelerate the widespread adoption of our products. We emphasize performance, attractive styling and the safety of our users and workforce in the design and manufacture of our products and are continuing to develop full self-driving technology for improved safety. We also strive to lower the cost of ownership for our customers through continuous efforts to reduce manufacturing costs and by offering financial and other services tailored to our products.",
        metadata={"year": 2023, "section": "business"}
    ),
    Document(
        id=2,
        page_content="We have previously experienced and may in the future experience launch and production ramp delays for new products and features. For example, we encountered unanticipated supplier issues that led to delays during the initial ramp of our first Model X and experienced challenges with a supplier and with ramping full automation for certain of our initial Model 3 manufacturing processes. In addition, we may introduce in the future new or unique manufacturing processes and design features for our products. As we expand our vehicle offerings and global footprint, there is no guarantee that we will be able to successfully and timely introduce and scale such processes or features.",
        metadata={"year": 2023, "section": "risk_factors"}
    ),
    Document(
        id=3,
        page_content="We recognize the importance of assessing, identifying, and managing material risks associated with cybersecurity threats, as such term is defined in Item 106(a) of Regulation S-K. These risks include, among other things: operational risks, intellectual property theft, fraud, extortion, harm to employees or customers and violation of data privacy or security laws. Identifying and assessing cybersecurity risk is integrated into our overall risk management systems and processes. Cybersecurity risks related to our business, technical operations, privacy and compliance issues are identified and addressed through a multi-faceted approach including third party assessments, internal IT Audit, IT security, governance, risk and compliance reviews. To defend, detect and respond to cybersecurity incidents, we, among other things: conduct proactive privacy and cybersecurity reviews of systems and applications, audit applicable data policies, perform penetration testing using external third-party tools and techniques to test security controls, operate a bug bounty program to encourage proactive vulnerability reporting, conduct employee training, monitor emerging laws and regulations related to data protection and information security (including our consumer products) and implement appropriate changes.",
        metadata={"year": 2023, "section": "cyber_security"}
    ),
    Document(
        id=4,
        page_content="The automotive segment includes the design, development, manufacturing, sales and leasing of high-performance fully electric vehicles as well as sales of automotive regulatory credits. Additionally, the automotive segment also includes services and other, which includes non-warranty after- sales vehicle services and parts, sales of used vehicles, retail merchandise, paid Supercharging and vehicle insurance revenue. The energy generation and storage segment includes the design, manufacture, installation, sales and leasing of solar energy generation and energy storage products and related services and sales of solar energy systems incentives.",
        metadata={"year": 2022, "section": "business"}
    ),
    Document(
        id=5,
        page_content="Since the first quarter of 2020, there has been a worldwide impact from the COVID-19 pandemic. Government regulations and shifting social behaviors have, at times, limited or closed non-essential transportation, government functions, business activities and person-to-person interactions. Global trade conditions and consumer trends that originated during the pandemic continue to persist and may also have long-lasting adverse impact on us and our industries independently of the progress of the pandemic.",
        metadata={"year": 2022, "section": "risk_factors"}
    ),
    Document(
        id=6,
        page_content="The German Umweltbundesamt issued our subsidiary in Germany a notice and fine in the amount of 12 million euro alleging its non-compliance under applicable laws relating to market participation notifications and take-back obligations with respect to end-of-life battery products required thereunder. In response to Tesla’s objection, the German Umweltbundesamt issued Tesla a revised fine notice dated April 29, 2021 in which it reduced the original fine amount to 1.45 million euro. This is primarily relating to administrative requirements, but Tesla has continued to take back battery packs, and filed a new objection in June 2021. A hearing took place on November 24, 2022, and the parties reached a settlement which resulted in a further reduction of the fine to 600,000 euro. Both parties have waived their right to appeal.",
        metadata={"year": 2022, "section": "legal_proceedings"}
    )
]

In [81]:
type(documents[0])

langchain_core.documents.base.Document

In [84]:
vectorstore = Chroma(
    collection_name="full_document_chunks",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./chroma_db" #to persist the database on disk
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [85]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023', 'full_document_chunks']

In [86]:
vectorstore.add_documents(
    documents=documents
)

['1', '2', '3', '4', '5', '6']

In [87]:
vectorstore


In [88]:
retriever = vectorstore.as_retriever(
    search_type="similarity", #type of search to perform
    search_kwargs={
        'k': 3  #number of similar documents to retrieve
    }
)

# Query Expansion

In [89]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [90]:
model_name = 'openai/gpt-oss-120b' #@param ["openai/gpt-oss-120b", "gpt-3.5-turbo", "gpt-4"]

In [91]:
user_input = "Any specific fines levied on the company in 2022?"

In [92]:
prompt = [
    {"role": "system", "content": query_expansion_system_message},
    {"role": "user", "content": user_message_template.format(question=user_input)}
]

In [93]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nAny specific fines levied on the company in 2022?\n</Question>\n'}]

In [94]:
query_expansions = client.chat.completions.create(model=model_name, 
                                                  messages=prompt, 
                                                  temperature=0)

In [95]:
type(query_expansions.choices[0].message.content)

str

In [96]:
print(query_expansions)

ChatCompletion(id='chatcmpl-d932222e-6c97-40a3-9425-dbd7bfc45bd0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='What fines were levied on the company in 2022?  \nWere any penalties or fines imposed on the company during 2022?  \nDid the company incur any regulatory fines in 2022?', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user wants query expansion: produce multiple versions of the question. Need at least 3 versions, each on new line, no numbering or bullets, no extra text. Provide variations: "What fines were imposed on the company in 2022?" "Were there any penalties assessed against the company in 2022?" "Did the company incur any regulatory fines in 2022?" etc. Ensure at least 3.', tool_calls=None))], created=1780373013, model='openai/gpt-oss-120b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_8a618bed98', usage=CompletionUsage

In [97]:
print(query_expansions.choices[-1].message.reasoning)

The user wants query expansion: produce multiple versions of the question. Need at least 3 versions, each on new line, no numbering or bullets, no extra text. Provide variations: "What fines were imposed on the company in 2022?" "Were there any penalties assessed against the company in 2022?" "Did the company incur any regulatory fines in 2022?" etc. Ensure at least 3.


In [98]:
# query_expansions_list = query_expansions.content.strip().split("\n")
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [99]:
query_expansions_list

['What fines were levied on the company in 2022?  ',
 'Were any penalties or fines imposed on the company during 2022?  ',
 'Did the company incur any regulatory fines in 2022?']

In [100]:
expanded_context_list = []

In [101]:
for query in query_expansions_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [102]:
len(expanded_context_list)

9

In [103]:
expanded_context_list

['The German Umweltbundesamt issued our subsidiary in Germany a notice and fine in the amount of 12 million euro alleging its non-compliance under applicable laws relating to market participation notifications and take-back obligations with respect to end-of-life battery products required thereunder. In response to Tesla’s objection, the German Umweltbundesamt issued Tesla a revised fine notice dated April 29, 2021 in which it reduced the original fine amount to 1.45 million euro. This is primarily relating to administrative requirements, but Tesla has continued to take back battery packs, and filed a new objection in June 2021. A hearing took place on November 24, 2022, and the parties reached a settlement which resulted in a further reduction of the fine to 600,000 euro. Both parties have waived their right to appeal.',
 'We recognize the importance of assessing, identifying, and managing material risks associated with cybersecurity threats, as such term is defined in Item 106(a) of 

In [104]:
final_context_documents = set(expanded_context_list)

In [105]:
len(final_context_documents)

4

In [106]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""



In [107]:
context_for_qna = "\n\n".join(map(str, final_context_documents))

In [108]:
prompts = [
    {"role": "system", "content": qna_system_message},
    {"role": "user", "content": qna_user_message_template.format(
        context=context_for_qna, 
        question=user_input)}
]

In [109]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nAny specific fines levied on the company in 2022?\n</Question>\n'}]

In [110]:
try:
    resonse = client.chat.completions.create(model=model_name, 
                                              messages=prompts, 
                                              temperature=0)
    prediction = resonse.choices[0].message.content.strip()

except Exception as e:
    prediction = f"Sorry, I encountered an error while trying to answer your question. Please try again later."


print(prediction)


The company faced a fine of €600,000 in 2022, resulting from a settlement after a hearing on November 24, 2022.


### Created by Vikash Kumar